# 05 — Multi-Source Vector Store and Retrieval Evaluation

## Responsible AI RAG Assistant

This notebook builds the multi-source vector store for the Responsible AI RAG Assistant.

The previous notebooks created:

- source inventory
- raw source files
- extracted source text
- multi-source chunk table
- source-level extraction quality checks

This notebook now converts the multi-source chunk table into embeddings and stores them in a local Chroma vector database.

## Main Goal

The goal is to create a stronger retrieval layer that can search across multiple authoritative public sources:

1. European Commission AI Act overview
2. Official EUR-Lex EU AI Act legal text
3. Official EUR-Lex GDPR legal text
4. NIST AI Risk Management Framework PDF
5. NIST AI RMF overview page

## Responsible-Use Note

This project is an educational and portfolio-ready AI Engineering prototype.

It does not provide legal advice.

The assistant should retrieve and summarize relevant public-source context, but real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.

Raw downloaded files, generated vector stores, `.env` files, and private documents should not be uploaded to GitHub.

In [1]:
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np

print("Notebook 05 setup started.")
print(f"Current date: {date.today()}")

Notebook 05 setup started.
Current date: 2026-07-02


In [2]:
# Define project paths

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "VECTOR_STORE_DIR": VECTOR_STORE_DIR,
    "REPORTS_DIR": REPORTS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
VECTOR_STORE_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\vector_store
Exists: True
--------------------------------------------------------------------------------
REPORTS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\reports
Exists: True
--------------------------------------------------------------------------------


## Load Multi-Source Chunk Table

The multi-source chunk table contains the retrieval-ready text chunks created in Notebook 04.

Each chunk includes:

- chunk ID
- source ID
- source name
- publisher
- source type
- topic metadata
- URL
- chunk text
- word count

In [3]:
# Load multi-source chunk table

multisource_chunks_path = PROCESSED_DATA_DIR / "multisource_chunks.csv"
chunk_summary_path = PROCESSED_DATA_DIR / "multisource_chunk_summary.csv"

multisource_chunks_df = pd.read_csv(multisource_chunks_path)
chunk_summary_by_source_df = pd.read_csv(chunk_summary_path)

print(f"Loaded multi-source chunks from: {multisource_chunks_path}")
print(f"Chunk table exists: {multisource_chunks_path.exists()}")
print(f"Number of chunks: {len(multisource_chunks_df)}")
print(f"Number of sources: {multisource_chunks_df['source_id'].nunique()}")
print(f"Columns: {list(multisource_chunks_df.columns)}")

multisource_chunks_df.head()

Loaded multi-source chunks from: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_chunks.csv
Chunk table exists: True
Number of chunks: 856
Number of sources: 5
Columns: ['chunk_id', 'source_id', 'source_name', 'publisher', 'source_type', 'main_topic', 'url', 'planned_use', 'raw_filename', 'processed_text_file', 'chunk_index', 'chunk_text', 'word_count', 'character_count', 'chunk_size_setting', 'chunk_overlap_setting']


,chunk_id,source_id,source_name,publisher,source_type,main_topic,url,planned_use,raw_filename,processed_text_file,chunk_index,chunk_text,word_count,character_count,chunk_size_setting,chunk_overlap_setting
0,SRC-001_CHUNK_0001,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,1,The AI Act is the first-ever legal framework o...,300,1745,300,60
1,SRC-001_CHUNK_0002,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,2,The AI Act ensures that Europeans can trust wh...,300,1970,300,60
2,SRC-001_CHUNK_0003,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,3,the practical application of the prohibited pr...,300,2051,300,60
3,SRC-001_CHUNK_0004,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,4,processes (e.g. AI solutions to prepare court ...,300,1896,300,60
4,SRC-001_CHUNK_0005,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...,src_001_european_commission_ai_act_overview.txt,src-001_extracted_text.txt,5,fall into this category. This includes applica...,300,1874,300,60


In [4]:
# Validate loaded multi-source chunks

required_columns = [
    "chunk_id",
    "source_id",
    "source_name",
    "publisher",
    "source_type",
    "main_topic",
    "url",
    "planned_use",
    "raw_filename",
    "processed_text_file",
    "chunk_index",
    "chunk_text",
    "word_count",
    "character_count",
]

missing_columns = [
    column for column in required_columns
    if column not in multisource_chunks_df.columns
]

empty_chunk_count = (
    multisource_chunks_df["chunk_text"].isna()
    | (multisource_chunks_df["chunk_text"].str.strip() == "")
).sum()

validation_summary = {
    "rows": len(multisource_chunks_df),
    "unique_sources": multisource_chunks_df["source_id"].nunique(),
    "missing_columns": missing_columns,
    "empty_chunk_count": int(empty_chunk_count),
    "min_words_per_chunk": int(multisource_chunks_df["word_count"].min()),
    "max_words_per_chunk": int(multisource_chunks_df["word_count"].max()),
    "average_words_per_chunk": round(multisource_chunks_df["word_count"].mean(), 1),
}

for key, value in validation_summary.items():
    print(f"{key}: {value}")

if missing_columns:
    print("WARNING: Missing required columns.")
elif empty_chunk_count > 0:
    print("WARNING: Empty chunks found.")
else:
    print("Loaded multi-source chunk table validation passed.")

rows: 856
unique_sources: 5
missing_columns: []
empty_chunk_count: 0
min_words_per_chunk: 69
max_words_per_chunk: 300
average_words_per_chunk: 299.0
Loaded multi-source chunk table validation passed.


In [5]:
# Display source-level chunk summary

chunk_summary_by_source_df

,source_id,source_name,publisher,chunk_count,total_chunk_words,average_chunk_words,min_chunk_words,max_chunk_words
0,SRC-001,AI Act,European Commission,10,2769,276.9,69,300
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,484,145034,299.7,134,300
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,292,87475,299.6,175,300
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,19967,298.0,167,300
4,SRC-005,AI Risk Management Framework overview page,NIST,3,682,227.3,82,300


## Generate Embeddings for Multi-Source Chunks

The next step is to convert all text chunks into numerical embedding vectors.

The project uses the local sentence-transformer model:

`sentence-transformers/all-MiniLM-L6-v2`

This model is suitable for a portfolio RAG project because it is:

- lightweight
- local
- reproducible
- fast enough for development
- appropriate for semantic retrieval demonstrations

The embeddings will later be stored in a Chroma vector database.

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {embedding_model_name}")

embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded successfully.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [7]:
# Test embedding generation on one sample chunk

sample_chunk_text = multisource_chunks_df.loc[0, "chunk_text"]

sample_embedding = embedding_model.encode(
    sample_chunk_text,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"Sample chunk ID: {multisource_chunks_df.loc[0, 'chunk_id']}")
print(f"Sample source: {multisource_chunks_df.loc[0, 'source_id']} — {multisource_chunks_df.loc[0, 'source_name']}")
print(f"Sample chunk word count: {multisource_chunks_df.loc[0, 'word_count']}")
print(f"Embedding type: {type(sample_embedding)}")
print(f"Embedding shape: {sample_embedding.shape}")
print(f"Embedding vector norm: {np.linalg.norm(sample_embedding):.4f}")
print("First 10 embedding values:")
print(sample_embedding[:10])

Sample chunk ID: SRC-001_CHUNK_0001
Sample source: SRC-001 — AI Act
Sample chunk word count: 300
Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
Embedding vector norm: 1.0000
First 10 embedding values:
[-3.8367860e-02 -6.3510746e-02  2.7370146e-03 -7.3454119e-02
  6.9103822e-02  6.9743745e-02  1.0670103e-02  4.6510965e-02
 -4.7601650e-05  5.6069471e-02]


In [8]:
# Generate embeddings for all multi-source chunks

chunk_texts = multisource_chunks_df["chunk_text"].astype(str).tolist()

print(f"Number of chunks to embed: {len(chunk_texts)}")
print(f"Embedding model: {embedding_model_name}")

multisource_embedding_matrix = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embedding generation completed.")
print(f"Embedding matrix type: {type(multisource_embedding_matrix)}")
print(f"Embedding matrix shape: {multisource_embedding_matrix.shape}")

Number of chunks to embed: 856
Embedding model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Embedding generation completed.
Embedding matrix type: <class 'numpy.ndarray'>
Embedding matrix shape: (856, 384)


In [9]:
# Validate embedding matrix

embedding_row_count = multisource_embedding_matrix.shape[0]
embedding_dimension = multisource_embedding_matrix.shape[1]

expected_row_count = len(multisource_chunks_df)

embedding_norms = np.linalg.norm(multisource_embedding_matrix, axis=1)

embedding_validation = {
    "expected_rows": expected_row_count,
    "embedding_rows": embedding_row_count,
    "embedding_dimension": embedding_dimension,
    "contains_nan": bool(np.isnan(multisource_embedding_matrix).any()),
    "contains_inf": bool(np.isinf(multisource_embedding_matrix).any()),
    "min_vector_norm": round(float(embedding_norms.min()), 4),
    "max_vector_norm": round(float(embedding_norms.max()), 4),
    "mean_vector_norm": round(float(embedding_norms.mean()), 4),
}

for key, value in embedding_validation.items():
    print(f"{key}: {value}")

if embedding_row_count != expected_row_count:
    print("WARNING: Number of embeddings does not match number of chunks.")
elif np.isnan(multisource_embedding_matrix).any():
    print("WARNING: Embedding matrix contains NaN values.")
elif np.isinf(multisource_embedding_matrix).any():
    print("WARNING: Embedding matrix contains infinite values.")
else:
    print("Embedding matrix validation passed.")

expected_rows: 856
embedding_rows: 856
embedding_dimension: 384
contains_nan: False
contains_inf: False
min_vector_norm: 1.0
max_vector_norm: 1.0
mean_vector_norm: 1.0
Embedding matrix validation passed.


In [10]:
# Save embeddings and metadata locally

embeddings_output_path = PROCESSED_DATA_DIR / "multisource_embeddings_all_minilm_l6_v2.npy"
embedding_metadata_output_path = PROCESSED_DATA_DIR / "multisource_embedding_metadata.csv"

np.save(embeddings_output_path, multisource_embedding_matrix)

embedding_metadata_df = multisource_chunks_df[
    [
        "chunk_id",
        "source_id",
        "source_name",
        "publisher",
        "source_type",
        "main_topic",
        "url",
        "chunk_index",
        "word_count",
        "character_count",
    ]
].copy()

embedding_metadata_df["embedding_model"] = embedding_model_name
embedding_metadata_df["embedding_row_index"] = range(len(embedding_metadata_df))

embedding_metadata_df.to_csv(
    embedding_metadata_output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved embedding matrix to: {embeddings_output_path}")
print(f"Embedding matrix file exists: {embeddings_output_path.exists()}")
print(f"Embedding matrix file size bytes: {embeddings_output_path.stat().st_size}")

print("-" * 80)

print(f"Saved embedding metadata to: {embedding_metadata_output_path}")
print(f"Embedding metadata file exists: {embedding_metadata_output_path.exists()}")
print(f"Embedding metadata rows: {len(embedding_metadata_df)}")

embedding_metadata_df.head()

Saved embedding matrix to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_embeddings_all_minilm_l6_v2.npy
Embedding matrix file exists: True
Embedding matrix file size bytes: 1314944
--------------------------------------------------------------------------------
Saved embedding metadata to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_embedding_metadata.csv
Embedding metadata file exists: True
Embedding metadata rows: 856


,chunk_id,source_id,source_name,publisher,source_type,main_topic,url,chunk_index,word_count,character_count,embedding_model,embedding_row_index
0,SRC-001_CHUNK_0001,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,1,300,1745,sentence-transformers/all-MiniLM-L6-v2,0
1,SRC-001_CHUNK_0002,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,2,300,1970,sentence-transformers/all-MiniLM-L6-v2,1
2,SRC-001_CHUNK_0003,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,3,300,2051,sentence-transformers/all-MiniLM-L6-v2,2
3,SRC-001_CHUNK_0004,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,4,300,1896,sentence-transformers/all-MiniLM-L6-v2,3
4,SRC-001_CHUNK_0005,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,5,300,1874,sentence-transformers/all-MiniLM-L6-v2,4


In [11]:
# Reload saved embeddings to confirm reproducibility

reloaded_embedding_matrix = np.load(embeddings_output_path)
reloaded_embedding_metadata_df = pd.read_csv(embedding_metadata_output_path)

print(f"Reloaded embedding matrix shape: {reloaded_embedding_matrix.shape}")
print(f"Reloaded metadata rows: {len(reloaded_embedding_metadata_df)}")

if reloaded_embedding_matrix.shape == multisource_embedding_matrix.shape and len(reloaded_embedding_metadata_df) == len(multisource_chunks_df):
    print("Embedding reload validation passed.")
else:
    print("WARNING: Reloaded embeddings or metadata do not match expected dimensions.")

Reloaded embedding matrix shape: (856, 384)
Reloaded metadata rows: 856
Embedding reload validation passed.


## Create Multi-Source Chroma Vector Store

The validated embedding matrix is now stored in a local Chroma vector database.

This replaces the earlier single-source vector store and creates a stronger retrieval layer across all five sources.

The vector store will contain:

- 856 text chunks
- normalized sentence-transformer embeddings
- source metadata
- source URLs
- chunk identifiers
- source names and publishers

The collection will be recreated during development to keep the notebook reproducible.

In [12]:
import chromadb


def clean_metadata_value(value):
    """
    Convert metadata values into Chroma-safe primitive values.
    Chroma metadata should not contain NaN, None, lists, or complex objects.
    """
    if pd.isna(value):
        return ""
    return str(value)


def build_chroma_metadata(row: pd.Series) -> dict:
    """
    Create clean metadata dictionary for one chunk.
    """
    return {
        "source_id": clean_metadata_value(row["source_id"]),
        "source_name": clean_metadata_value(row["source_name"]),
        "publisher": clean_metadata_value(row["publisher"]),
        "source_type": clean_metadata_value(row["source_type"]),
        "main_topic": clean_metadata_value(row["main_topic"]),
        "url": clean_metadata_value(row["url"]),
        "planned_use": clean_metadata_value(row["planned_use"]),
        "raw_filename": clean_metadata_value(row["raw_filename"]),
        "processed_text_file": clean_metadata_value(row["processed_text_file"]),
        "chunk_index": int(row["chunk_index"]),
        "word_count": int(row["word_count"]),
        "character_count": int(row["character_count"]),
        "embedding_model": embedding_model_name,
    }


chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

multisource_collection_name = "responsible_ai_rag_multisource_v1"

# Recreate collection for reproducibility during development
try:
    chroma_client.delete_collection(name=multisource_collection_name)
    print(f"Deleted existing collection: {multisource_collection_name}")
except Exception:
    print(f"No existing collection found with name: {multisource_collection_name}")

multisource_collection = chroma_client.create_collection(
    name=multisource_collection_name,
    metadata={
        "description": "Multi-source Responsible AI RAG chunks from EU AI Act, GDPR, and NIST AI RMF sources",
        "embedding_model": embedding_model_name,
        "source_count": str(multisource_chunks_df["source_id"].nunique()),
        "chunk_count": str(len(multisource_chunks_df)),
    },
)

print(f"Created Chroma collection: {multisource_collection_name}")

No existing collection found with name: responsible_ai_rag_multisource_v1
Created Chroma collection: responsible_ai_rag_multisource_v1


In [13]:
# Prepare records for Chroma insertion

chroma_ids = multisource_chunks_df["chunk_id"].astype(str).tolist()
chroma_documents = multisource_chunks_df["chunk_text"].astype(str).tolist()

chroma_metadatas = [
    build_chroma_metadata(row)
    for _, row in multisource_chunks_df.iterrows()
]

chroma_embeddings = multisource_embedding_matrix.tolist()

print(f"Prepared IDs: {len(chroma_ids)}")
print(f"Prepared documents: {len(chroma_documents)}")
print(f"Prepared metadata records: {len(chroma_metadatas)}")
print(f"Prepared embeddings: {len(chroma_embeddings)}")

print("-" * 80)

print(f"Example ID: {chroma_ids[0]}")
print(f"Example metadata: {chroma_metadatas[0]}")
print(f"Example document preview: {chroma_documents[0][:300]}")

Prepared IDs: 856
Prepared documents: 856
Prepared metadata records: 856
Prepared embeddings: 856
--------------------------------------------------------------------------------
Example ID: SRC-001_CHUNK_0001
Example metadata: {'source_id': 'SRC-001', 'source_name': 'AI Act', 'publisher': 'European Commission', 'source_type': 'Official web page', 'main_topic': 'EU AI Act overview and risk-based approach', 'url': 'https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai', 'planned_use': 'Governance overview and user-facing source explanation', 'raw_filename': 'src_001_european_commission_ai_act_overview.txt', 'processed_text_file': 'src-001_extracted_text.txt', 'chunk_index': 1, 'word_count': 300, 'character_count': 1745, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2'}
Example document preview: The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU

In [14]:
# Add all chunks to Chroma collection in batches

batch_size = 100
total_records = len(chroma_ids)

for start_idx in range(0, total_records, batch_size):
    end_idx = min(start_idx + batch_size, total_records)

    multisource_collection.add(
        ids=chroma_ids[start_idx:end_idx],
        documents=chroma_documents[start_idx:end_idx],
        metadatas=chroma_metadatas[start_idx:end_idx],
        embeddings=chroma_embeddings[start_idx:end_idx],
    )

    print(f"Added records {start_idx + 1} to {end_idx} of {total_records}")

print("-" * 80)
print("All records added to Chroma collection.")
print(f"Collection count: {multisource_collection.count()}")

Added records 1 to 100 of 856
Added records 101 to 200 of 856
Added records 201 to 300 of 856
Added records 301 to 400 of 856
Added records 401 to 500 of 856
Added records 501 to 600 of 856
Added records 601 to 700 of 856
Added records 701 to 800 of 856
Added records 801 to 856 of 856
--------------------------------------------------------------------------------
All records added to Chroma collection.
Collection count: 856


In [15]:
# Validate Chroma collection

collection_count = multisource_collection.count()
expected_count = len(multisource_chunks_df)

print(f"Expected chunk count: {expected_count}")
print(f"Chroma collection count: {collection_count}")

if collection_count == expected_count:
    print("Chroma collection validation passed.")
else:
    print("WARNING: Chroma collection count does not match expected chunk count.")

Expected chunk count: 856
Chroma collection count: 856
Chroma collection validation passed.


In [16]:
# Peek at a few records from the Chroma collection

collection_peek = multisource_collection.peek(limit=3)

print(f"Returned IDs: {collection_peek['ids']}")
print("-" * 80)

for index, document in enumerate(collection_peek["documents"]):
    print(f"Record {index + 1}")
    print(f"ID: {collection_peek['ids'][index]}")
    print(f"Metadata: {collection_peek['metadatas'][index]}")
    print(f"Document preview: {document[:400]}")
    print("-" * 80)

Returned IDs: ['SRC-001_CHUNK_0001', 'SRC-001_CHUNK_0002', 'SRC-001_CHUNK_0003']
--------------------------------------------------------------------------------
Record 1
ID: SRC-001_CHUNK_0001
Metadata: {'main_topic': 'EU AI Act overview and risk-based approach', 'source_name': 'AI Act', 'planned_use': 'Governance overview and user-facing source explanation', 'processed_text_file': 'src-001_extracted_text.txt', 'chunk_index': 1, 'source_type': 'Official web page', 'character_count': 1745, 'publisher': 'European Commission', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'url': 'https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai', 'source_id': 'SRC-001', 'raw_filename': 'src_001_european_commission_ai_act_overview.txt', 'word_count': 300}
Document preview: The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU) 2024/1689 laying down harmonis

## Multi-Source Retrieval Function

This section creates a reusable retrieval function for the multi-source Chroma vector store.

The function:

- embeds the user query
- searches the Chroma collection
- retrieves the most relevant chunks
- returns source metadata, distances, and retrieved text
- stores results in a readable DataFrame

This retrieval layer is the foundation for the final RAG assistant.

In [17]:
def search_multisource_vector_store(query: str, n_results: int = 5) -> pd.DataFrame:
    """
    Search the multi-source Chroma vector store using a natural-language query.
    """
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    results = multisource_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    retrieved_records = []

    for rank, chunk_id in enumerate(results["ids"][0], start=1):
        metadata = results["metadatas"][0][rank - 1]
        document = results["documents"][0][rank - 1]
        distance = results["distances"][0][rank - 1]

        retrieved_records.append(
            {
                "rank": rank,
                "chunk_id": chunk_id,
                "distance": distance,
                "source_id": metadata.get("source_id", ""),
                "source_name": metadata.get("source_name", ""),
                "publisher": metadata.get("publisher", ""),
                "source_type": metadata.get("source_type", ""),
                "main_topic": metadata.get("main_topic", ""),
                "chunk_index": metadata.get("chunk_index", ""),
                "word_count": metadata.get("word_count", ""),
                "url": metadata.get("url", ""),
                "retrieved_text": document,
            }
        )

    return pd.DataFrame(retrieved_records)


print("Multi-source retrieval function is ready.")

Multi-source retrieval function is ready.


In [18]:
# Test one multi-source retrieval query

test_query = "What is the risk-based approach of the EU AI Act?"

test_query_results_df = search_multisource_vector_store(
    query=test_query,
    n_results=5,
)

print(f"Query: {test_query}")
print(f"Retrieved chunks: {len(test_query_results_df)}")

test_query_results_df[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "publisher",
        "chunk_index",
        "word_count",
        "retrieved_text",
    ]
]

Query: What is the risk-based approach of the EU AI Act?
Retrieved chunks: 5


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
0,1,0.525325,SRC-001,AI Act,European Commission,2,300,The AI Act ensures that Europeans can trust wh...
1,2,0.537072,SRC-001,AI Act,European Commission,1,300,The AI Act is the first-ever legal framework o...
2,3,0.620605,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,167,or aligned with other approaches to managing A...
3,4,0.731535,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,20,300,Key Dimensions of an AI System. Modified from ...
4,5,0.735104,SRC-001,AI Act,European Commission,8,300,set for the rules governing high-risk AI syste...


## Retrieval Evaluation Test Set

A stronger RAG project should not only build a vector store; it should also evaluate whether retrieval works across different source domains.

The test set below checks retrieval coverage for:

- EU AI Act overview
- official AI Act legal text
- GDPR legal text
- NIST AI Risk Management Framework
- transparency and human oversight topics
- data protection and risk management topics

This makes the project more credible for recruiters because it shows retrieval quality control, not only implementation.

In [19]:
# Multi-source retrieval evaluation questions

retrieval_test_questions = [
    {
        "test_id": "MSR-001",
        "question": "What is the risk-based approach of the EU AI Act?",
        "expected_sources": ["SRC-001", "SRC-002"],
        "topic": "EU AI Act risk-based approach",
    },
    {
        "test_id": "MSR-002",
        "question": "What obligations apply to high-risk AI systems?",
        "expected_sources": ["SRC-002", "SRC-001"],
        "topic": "High-risk AI obligations",
    },
    {
        "test_id": "MSR-003",
        "question": "What does the AI Act say about transparency for AI-generated content?",
        "expected_sources": ["SRC-002", "SRC-001"],
        "topic": "AI transparency",
    },
    {
        "test_id": "MSR-004",
        "question": "What is personal data under the GDPR?",
        "expected_sources": ["SRC-003"],
        "topic": "GDPR personal data",
    },
    {
        "test_id": "MSR-005",
        "question": "What are the GDPR principles for processing personal data?",
        "expected_sources": ["SRC-003"],
        "topic": "GDPR processing principles",
    },
    {
        "test_id": "MSR-006",
        "question": "What does NIST say about managing AI risks?",
        "expected_sources": ["SRC-004", "SRC-005"],
        "topic": "NIST AI risk management",
    },
    {
        "test_id": "MSR-007",
        "question": "What are the NIST AI RMF functions Govern, Map, Measure, and Manage?",
        "expected_sources": ["SRC-004", "SRC-005"],
        "topic": "NIST AI RMF functions",
    },
    {
        "test_id": "MSR-008",
        "question": "How should organizations think about accountability, transparency, and human oversight in AI governance?",
        "expected_sources": ["SRC-001", "SRC-002", "SRC-004"],
        "topic": "Responsible AI governance",
    },
]

retrieval_test_questions_df = pd.DataFrame(retrieval_test_questions)

retrieval_test_questions_df

,test_id,question,expected_sources,topic
0,MSR-001,What is the risk-based approach of the EU AI Act?,"[SRC-001, SRC-002]",EU AI Act risk-based approach
1,MSR-002,What obligations apply to high-risk AI systems?,"[SRC-002, SRC-001]",High-risk AI obligations
2,MSR-003,What does the AI Act say about transparency fo...,"[SRC-002, SRC-001]",AI transparency
3,MSR-004,What is personal data under the GDPR?,[SRC-003],GDPR personal data
4,MSR-005,What are the GDPR principles for processing pe...,[SRC-003],GDPR processing principles
5,MSR-006,What does NIST say about managing AI risks?,"[SRC-004, SRC-005]",NIST AI risk management
6,MSR-007,"What are the NIST AI RMF functions Govern, Map...","[SRC-004, SRC-005]",NIST AI RMF functions
7,MSR-008,How should organizations think about accountab...,"[SRC-001, SRC-002, SRC-004]",Responsible AI governance


In [20]:
# Run retrieval evaluation over the multi-source test set

retrieval_evaluation_records = []
retrieval_detailed_records = []

N_RESULTS_FOR_EVALUATION = 5

for _, test_row in retrieval_test_questions_df.iterrows():
    test_id = test_row["test_id"]
    question = test_row["question"]
    expected_sources = test_row["expected_sources"]
    topic = test_row["topic"]

    results_df = search_multisource_vector_store(
        query=question,
        n_results=N_RESULTS_FOR_EVALUATION,
    )

    retrieved_sources = results_df["source_id"].tolist()
    unique_retrieved_sources = sorted(results_df["source_id"].unique().tolist())

    expected_source_hit = any(
        source_id in retrieved_sources
        for source_id in expected_sources
    )

    top_source_id = results_df.loc[0, "source_id"]
    top_source_name = results_df.loc[0, "source_name"]
    top_distance = results_df.loc[0, "distance"]
    top_chunk_id = results_df.loc[0, "chunk_id"]

    if expected_source_hit and top_source_id in expected_sources:
        retrieval_quality = "strong"
    elif expected_source_hit:
        retrieval_quality = "acceptable"
    else:
        retrieval_quality = "needs_review"

    retrieval_evaluation_records.append(
        {
            "test_id": test_id,
            "topic": topic,
            "question": question,
            "expected_sources": ", ".join(expected_sources),
            "top_source_id": top_source_id,
            "top_source_name": top_source_name,
            "top_chunk_id": top_chunk_id,
            "top_distance": top_distance,
            "unique_retrieved_sources": ", ".join(unique_retrieved_sources),
            "expected_source_hit": expected_source_hit,
            "retrieval_quality": retrieval_quality,
        }
    )

    for _, result_row in results_df.iterrows():
        retrieval_detailed_records.append(
            {
                "test_id": test_id,
                "topic": topic,
                "question": question,
                "rank": result_row["rank"],
                "distance": result_row["distance"],
                "chunk_id": result_row["chunk_id"],
                "source_id": result_row["source_id"],
                "source_name": result_row["source_name"],
                "publisher": result_row["publisher"],
                "chunk_index": result_row["chunk_index"],
                "word_count": result_row["word_count"],
                "url": result_row["url"],
                "retrieved_text": result_row["retrieved_text"],
            }
        )

retrieval_evaluation_df = pd.DataFrame(retrieval_evaluation_records)
retrieval_detailed_results_df = pd.DataFrame(retrieval_detailed_records)

retrieval_evaluation_df

,test_id,topic,question,expected_sources,top_source_id,top_source_name,top_chunk_id,top_distance,unique_retrieved_sources,expected_source_hit,retrieval_quality
0,MSR-001,EU AI Act risk-based approach,What is the risk-based approach of the EU AI Act?,"SRC-001, SRC-002",SRC-001,AI Act,SRC-001_CHUNK_0002,0.525325,"SRC-001, SRC-004",True,strong
1,MSR-002,High-risk AI obligations,What obligations apply to high-risk AI systems?,"SRC-002, SRC-001",SRC-004,Artificial Intelligence Risk Management Framew...,SRC-004_CHUNK_0005,0.577118,"SRC-001, SRC-004",True,acceptable
2,MSR-003,AI transparency,What does the AI Act say about transparency fo...,"SRC-002, SRC-001",SRC-001,AI Act,SRC-001_CHUNK_0004,0.550183,"SRC-001, SRC-004",True,strong
3,MSR-004,GDPR personal data,What is personal data under the GDPR?,SRC-003,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,SRC-002_CHUNK_0241,0.775993,"SRC-002, SRC-003",True,acceptable
4,MSR-005,GDPR processing principles,What are the GDPR principles for processing pe...,SRC-003,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,SRC-002_CHUNK_0241,0.660177,"SRC-002, SRC-003",True,acceptable
5,MSR-006,NIST AI risk management,What does NIST say about managing AI risks?,"SRC-004, SRC-005",SRC-005,AI Risk Management Framework overview page,SRC-005_CHUNK_0003,0.403003,"SRC-004, SRC-005",True,strong
6,MSR-007,NIST AI RMF functions,"What are the NIST AI RMF functions Govern, Map...","SRC-004, SRC-005",SRC-004,Artificial Intelligence Risk Management Framew...,SRC-004_CHUNK_0034,0.636821,SRC-004,True,strong
7,MSR-008,Responsible AI governance,How should organizations think about accountab...,"SRC-001, SRC-002, SRC-004",SRC-004,Artificial Intelligence Risk Management Framew...,SRC-004_CHUNK_0058,0.726825,SRC-004,True,strong


In [21]:
# Retrieval evaluation summary

print("Retrieval quality summary:")
print(retrieval_evaluation_df["retrieval_quality"].value_counts(dropna=False))

print("-" * 80)

print("Expected source hit summary:")
print(retrieval_evaluation_df["expected_source_hit"].value_counts(dropna=False))

retrieval_evaluation_df[
    [
        "test_id",
        "topic",
        "top_source_id",
        "top_source_name",
        "top_distance",
        "unique_retrieved_sources",
        "expected_source_hit",
        "retrieval_quality",
    ]
]

Retrieval quality summary:
retrieval_quality
strong        5
acceptable    3
Name: count, dtype: int64
--------------------------------------------------------------------------------
Expected source hit summary:
expected_source_hit
True    8
Name: count, dtype: int64


,test_id,topic,top_source_id,top_source_name,top_distance,unique_retrieved_sources,expected_source_hit,retrieval_quality
0,MSR-001,EU AI Act risk-based approach,SRC-001,AI Act,0.525325,"SRC-001, SRC-004",True,strong
1,MSR-002,High-risk AI obligations,SRC-004,Artificial Intelligence Risk Management Framew...,0.577118,"SRC-001, SRC-004",True,acceptable
2,MSR-003,AI transparency,SRC-001,AI Act,0.550183,"SRC-001, SRC-004",True,strong
3,MSR-004,GDPR personal data,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,0.775993,"SRC-002, SRC-003",True,acceptable
4,MSR-005,GDPR processing principles,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,0.660177,"SRC-002, SRC-003",True,acceptable
5,MSR-006,NIST AI risk management,SRC-005,AI Risk Management Framework overview page,0.403003,"SRC-004, SRC-005",True,strong
6,MSR-007,NIST AI RMF functions,SRC-004,Artificial Intelligence Risk Management Framew...,0.636821,SRC-004,True,strong
7,MSR-008,Responsible AI governance,SRC-004,Artificial Intelligence Risk Management Framew...,0.726825,SRC-004,True,strong


In [22]:
# Inspect detailed retrieval results for all test questions

for test_id in retrieval_test_questions_df["test_id"]:
    subset_df = retrieval_detailed_results_df[
        retrieval_detailed_results_df["test_id"] == test_id
    ].copy()

    question = subset_df["question"].iloc[0]

    print("=" * 100)
    print(f"TEST ID: {test_id}")
    print(f"QUESTION: {question}")
    print("-" * 100)

    display(
        subset_df[
            [
                "rank",
                "distance",
                "source_id",
                "source_name",
                "publisher",
                "chunk_index",
                "word_count",
                "retrieved_text",
            ]
        ]
    )

TEST ID: MSR-001
QUESTION: What is the risk-based approach of the EU AI Act?
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
0,1,0.525325,SRC-001,AI Act,European Commission,2,300,The AI Act ensures that Europeans can trust wh...
1,2,0.537072,SRC-001,AI Act,European Commission,1,300,The AI Act is the first-ever legal framework o...
2,3,0.620605,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,167,or aligned with other approaches to managing A...
3,4,0.731535,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,20,300,Key Dimensions of an AI System. Modified from ...
4,5,0.735104,SRC-001,AI Act,European Commission,8,300,set for the rules governing high-risk AI syste...


TEST ID: MSR-002
QUESTION: What obligations apply to high-risk AI systems?
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
5,1,0.577118,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,5,300,myriad standards and best practices to help or...
6,2,0.603582,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,4,300,is shown as the base for other trustworthi- ne...
7,3,0.615468,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,167,or aligned with other approaches to managing A...
8,4,0.639720,SRC-001,AI Act,European Commission,2,300,The AI Act ensures that Europeans can trust wh...
9,5,0.640791,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,63,300,should be monitored and deployed to take advan...


TEST ID: MSR-003
QUESTION: What does the AI Act say about transparency for AI-generated content?
----------------------------------------------------------------------------------------------------


...

,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
25,1,0.403003,SRC-005,AI Risk Management Framework overview page,NIST,3,82,"FinRegLab, and the Stanford Institute for Huma..."
26,2,0.544553,SRC-005,AI Risk Management Framework overview page,NIST,2,300,aligns with their goals and priorities. On Apr...
27,3,0.610519,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,9,300,and the Plan for Federal Engagement in Develop...
28,4,0.634200,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,167,or aligned with other approaches to managing A...
29,5,0.652685,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,33,300,in Artificial Intelligence.) Page 18 [Page 24]...


TEST ID: MSR-007
QUESTION: What are the NIST AI RMF functions Govern, Map, Measure, and Manage?
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
30,1,0.636821,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,34,300,for increased awareness of downstream risks; •...
31,2,0.751052,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,36,300,there are categories and subcategories with el...
32,3,0.758921,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,33,300,in Artificial Intelligence.) Page 18 [Page 24]...
33,4,0.765745,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,8,300,"the AI RMF is put into use, additional lessons..."
34,5,0.790972,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,35,300,"AI RMF 1.0 deployed, or evaluated – which can ..."


TEST ID: MSR-008
QUESTION: How should organizations think about accountability, transparency, and human oversight in AI governance?
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,retrieved_text
35,1,0.726825,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,58,300,and AI impact assessment teams. AI Impact Asse...
36,2,0.791689,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,28,300,on the stage of the AI lifecycle and tailored ...
37,3,0.821112,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,4,300,is shown as the base for other trustworthi- ne...
38,4,0.837066,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,64,300,systems may specifically require human oversig...
39,5,0.858993,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,39,300,"and deployment. GOVERN 3: Workforce diversity,..."


In [23]:
# Save retrieval evaluation results

retrieval_evaluation_output_path = PROCESSED_DATA_DIR / "multisource_retrieval_evaluation_summary.csv"
retrieval_detailed_output_path = PROCESSED_DATA_DIR / "multisource_retrieval_detailed_results.csv"

retrieval_evaluation_df.to_csv(
    retrieval_evaluation_output_path,
    index=False,
    encoding="utf-8",
)

retrieval_detailed_results_df.to_csv(
    retrieval_detailed_output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved retrieval evaluation summary to: {retrieval_evaluation_output_path}")
print(f"Summary file exists: {retrieval_evaluation_output_path.exists()}")
print(f"Summary rows: {len(retrieval_evaluation_df)}")

print("-" * 80)

print(f"Saved detailed retrieval results to: {retrieval_detailed_output_path}")
print(f"Detailed file exists: {retrieval_detailed_output_path.exists()}")
print(f"Detailed rows: {len(retrieval_detailed_results_df)}")

Saved retrieval evaluation summary to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_retrieval_evaluation_summary.csv
Summary file exists: True
Summary rows: 8
--------------------------------------------------------------------------------
Saved detailed retrieval results to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\multisource_retrieval_detailed_results.csv
Detailed file exists: True
Detailed rows: 40


## Source-Aware Retrieval Improvement

The baseline multi-source retrieval works well, but some questions can retrieve legally related sources that are semantically similar but not the best primary source.

For example:

- GDPR questions may retrieve the EU AI Act first because the AI Act references data protection concepts.
- NIST questions may retrieve AI Act governance chunks because both discuss AI risk and oversight.

To improve retrieval quality, this section adds a lightweight source-aware reranking layer.

The goal is not to hard-code answers, but to improve retrieval precision by identifying the likely source family:

- GDPR questions → prioritize SRC-003
- NIST / RMF questions → prioritize SRC-004 and SRC-005
- EU AI Act questions → prioritize SRC-001 and SRC-002

This is useful in a real RAG system because legal and governance corpora often contain overlapping terminology.

In [24]:
def infer_preferred_sources(query: str) -> list[str]:
    """
    Infer likely preferred source families from the user query.

    This is a lightweight routing/reranking helper, not an answer generator.
    It helps prioritize the most authoritative source family when multiple
    legal/governance sources contain overlapping terminology.
    """
    query_lower = query.lower()

    gdpr_keywords = [
        "gdpr",
        "personal data",
        "data subject",
        "data protection",
        "lawful basis",
        "processing personal data",
        "controller",
        "processor",
    ]

    nist_keywords = [
        "nist",
        "ai rmf",
        "risk management framework",
        "govern",
        "map",
        "measure",
        "manage",
    ]

    ai_act_keywords = [
        "ai act",
        "high-risk",
        "high risk",
        "prohibited",
        "transparency",
        "ai-generated",
        "general-purpose ai",
        "risk-based approach",
    ]

    if any(keyword in query_lower for keyword in gdpr_keywords):
        return ["SRC-003"]

    if any(keyword in query_lower for keyword in nist_keywords):
        return ["SRC-004", "SRC-005"]

    if any(keyword in query_lower for keyword in ai_act_keywords):
        return ["SRC-001", "SRC-002"]

    return []


print("Source-family detection function is ready.")

Source-family detection function is ready.


In [25]:
def search_multisource_vector_store_source_aware(
    query: str,
    n_results: int = 5,
    n_initial_results: int = 20,
) -> pd.DataFrame:
    """
    Search the multi-source vector store and apply lightweight source-aware reranking.

    The function first retrieves a broader candidate set from Chroma.
    Then, if preferred source families are detected, it prioritizes chunks
    from those sources while preserving semantic distance order.
    """
    preferred_sources = infer_preferred_sources(query)

    initial_results_df = search_multisource_vector_store(
        query=query,
        n_results=n_initial_results,
    )

    initial_results_df["preferred_source"] = initial_results_df["source_id"].isin(
        preferred_sources
    )

    if preferred_sources and initial_results_df["preferred_source"].any():
        reranked_df = initial_results_df.sort_values(
            by=["preferred_source", "distance"],
            ascending=[False, True],
        ).head(n_results)
    else:
        reranked_df = initial_results_df.sort_values(
            by="distance",
            ascending=True,
        ).head(n_results)

    reranked_df = reranked_df.copy()
    reranked_df["rank"] = range(1, len(reranked_df) + 1)
    reranked_df["preferred_sources"] = ", ".join(preferred_sources)

    return reranked_df


print("Source-aware retrieval function is ready.")

Source-aware retrieval function is ready.


In [26]:
# Test source-aware retrieval on a GDPR-specific question

gdpr_test_query = "What is personal data under the GDPR?"

gdpr_source_aware_results_df = search_multisource_vector_store_source_aware(
    query=gdpr_test_query,
    n_results=5,
    n_initial_results=20,
)

print(f"Query: {gdpr_test_query}")
print(f"Preferred sources: {infer_preferred_sources(gdpr_test_query)}")

gdpr_source_aware_results_df[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "publisher",
        "chunk_index",
        "word_count",
        "preferred_source",
        "retrieved_text",
    ]
]

Query: What is personal data under the GDPR?
Preferred sources: ['SRC-003']


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source,retrieved_text
1,1,0.807489,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,56,300,True,ies of personal data should be allo wed only u...
2,2,0.813962,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,4,300,True,of personal data. The exc hange of personal da...
3,3,0.815230,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,133,300,True,data consisting of the use of personal data to...
4,4,0.841520,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,132,300,True,U nion. 3. This Regulation applies to the proc...
5,5,0.885930,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,3,300,True,the Offi cial Jour nal). P osition of the Euro...


In [27]:
# Run source-aware retrieval evaluation over the same test set

source_aware_evaluation_records = []

for _, test_row in retrieval_test_questions_df.iterrows():
    test_id = test_row["test_id"]
    question = test_row["question"]
    expected_sources = test_row["expected_sources"]
    topic = test_row["topic"]

    results_df = search_multisource_vector_store_source_aware(
        query=question,
        n_results=5,
        n_initial_results=20,
    )

    retrieved_sources = results_df["source_id"].tolist()
    unique_retrieved_sources = sorted(results_df["source_id"].unique().tolist())

    expected_source_hit = any(
        source_id in retrieved_sources
        for source_id in expected_sources
    )

    top_source_id = results_df.loc[results_df.index[0], "source_id"]
    top_source_name = results_df.loc[results_df.index[0], "source_name"]
    top_distance = results_df.loc[results_df.index[0], "distance"]
    top_chunk_id = results_df.loc[results_df.index[0], "chunk_id"]
    preferred_sources = infer_preferred_sources(question)

    if expected_source_hit and top_source_id in expected_sources:
        retrieval_quality = "strong"
    elif expected_source_hit:
        retrieval_quality = "acceptable"
    else:
        retrieval_quality = "needs_review"

    source_aware_evaluation_records.append(
        {
            "test_id": test_id,
            "topic": topic,
            "question": question,
            "expected_sources": ", ".join(expected_sources),
            "preferred_sources": ", ".join(preferred_sources),
            "top_source_id": top_source_id,
            "top_source_name": top_source_name,
            "top_chunk_id": top_chunk_id,
            "top_distance": top_distance,
            "unique_retrieved_sources": ", ".join(unique_retrieved_sources),
            "expected_source_hit": expected_source_hit,
            "retrieval_quality": retrieval_quality,
        }
    )

source_aware_retrieval_evaluation_df = pd.DataFrame(source_aware_evaluation_records)

source_aware_retrieval_evaluation_df

,test_id,topic,question,expected_sources,preferred_sources,top_source_id,top_source_name,top_chunk_id,top_distance,unique_retrieved_sources,expected_source_hit,retrieval_quality
0,MSR-001,EU AI Act risk-based approach,What is the risk-based approach of the EU AI Act?,"SRC-001, SRC-002","SRC-001, SRC-002",SRC-001,AI Act,SRC-001_CHUNK_0002,0.525325,SRC-001,True,strong
1,MSR-002,High-risk AI obligations,What obligations apply to high-risk AI systems?,"SRC-002, SRC-001","SRC-001, SRC-002",SRC-001,AI Act,SRC-001_CHUNK_0002,0.639720,"SRC-001, SRC-004",True,strong
2,MSR-003,AI transparency,What does the AI Act say about transparency fo...,"SRC-002, SRC-001","SRC-001, SRC-002",SRC-001,AI Act,SRC-001_CHUNK_0004,0.550183,"SRC-001, SRC-002",True,strong
3,MSR-004,GDPR personal data,What is personal data under the GDPR?,SRC-003,SRC-003,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,SRC-003_CHUNK_0056,0.807489,SRC-003,True,strong
4,MSR-005,GDPR processing principles,What are the GDPR principles for processing pe...,SRC-003,SRC-003,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,SRC-003_CHUNK_0056,0.663028,SRC-003,True,strong
5,MSR-006,NIST AI risk management,What does NIST say about managing AI risks?,"SRC-004, SRC-005","SRC-004, SRC-005",SRC-005,AI Risk Management Framework overview page,SRC-005_CHUNK_0003,0.403003,"SRC-004, SRC-005",True,strong
6,MSR-007,NIST AI RMF functions,"What are the NIST AI RMF functions Govern, Map...","SRC-004, SRC-005","SRC-004, SRC-005",SRC-004,Artificial Intelligence Risk Management Framew...,SRC-004_CHUNK_0034,0.636821,SRC-004,True,strong
7,MSR-008,Responsible AI governance,How should organizations think about accountab...,"SRC-001, SRC-002, SRC-004","SRC-004, SRC-005",SRC-004,Artificial Intelligence Risk Management Framew...,SRC-004_CHUNK_0058,0.726825,SRC-004,True,strong


In [28]:
# Compare baseline retrieval and source-aware retrieval

baseline_comparison_df = retrieval_evaluation_df[
    [
        "test_id",
        "topic",
        "top_source_id",
        "retrieval_quality",
        "expected_source_hit",
    ]
].rename(
    columns={
        "top_source_id": "baseline_top_source_id",
        "retrieval_quality": "baseline_quality",
        "expected_source_hit": "baseline_expected_source_hit",
    }
)

source_aware_comparison_df = source_aware_retrieval_evaluation_df[
    [
        "test_id",
        "top_source_id",
        "retrieval_quality",
        "expected_source_hit",
        "preferred_sources",
    ]
].rename(
    columns={
        "top_source_id": "source_aware_top_source_id",
        "retrieval_quality": "source_aware_quality",
        "expected_source_hit": "source_aware_expected_source_hit",
    }
)

retrieval_comparison_df = baseline_comparison_df.merge(
    source_aware_comparison_df,
    on="test_id",
    how="left",
)

retrieval_comparison_df

,test_id,topic,baseline_top_source_id,baseline_quality,baseline_expected_source_hit,source_aware_top_source_id,source_aware_quality,source_aware_expected_source_hit,preferred_sources
0,MSR-001,EU AI Act risk-based approach,SRC-001,strong,True,SRC-001,strong,True,"SRC-001, SRC-002"
1,MSR-002,High-risk AI obligations,SRC-004,acceptable,True,SRC-001,strong,True,"SRC-001, SRC-002"
2,MSR-003,AI transparency,SRC-001,strong,True,SRC-001,strong,True,"SRC-001, SRC-002"
3,MSR-004,GDPR personal data,SRC-002,acceptable,True,SRC-003,strong,True,SRC-003
4,MSR-005,GDPR processing principles,SRC-002,acceptable,True,SRC-003,strong,True,SRC-003
5,MSR-006,NIST AI risk management,SRC-005,strong,True,SRC-005,strong,True,"SRC-004, SRC-005"
6,MSR-007,NIST AI RMF functions,SRC-004,strong,True,SRC-004,strong,True,"SRC-004, SRC-005"
7,MSR-008,Responsible AI governance,SRC-004,strong,True,SRC-004,strong,True,"SRC-004, SRC-005"


In [29]:
# Save source-aware retrieval evaluation and comparison

source_aware_evaluation_output_path = PROCESSED_DATA_DIR / "source_aware_retrieval_evaluation_summary.csv"
retrieval_comparison_output_path = PROCESSED_DATA_DIR / "retrieval_baseline_vs_source_aware_comparison.csv"

source_aware_retrieval_evaluation_df.to_csv(
    source_aware_evaluation_output_path,
    index=False,
    encoding="utf-8",
)

retrieval_comparison_df.to_csv(
    retrieval_comparison_output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved source-aware evaluation to: {source_aware_evaluation_output_path}")
print(f"Source-aware evaluation exists: {source_aware_evaluation_output_path.exists()}")
print(f"Source-aware evaluation rows: {len(source_aware_retrieval_evaluation_df)}")

print("-" * 80)

print(f"Saved retrieval comparison to: {retrieval_comparison_output_path}")
print(f"Retrieval comparison exists: {retrieval_comparison_output_path.exists()}")
print(f"Retrieval comparison rows: {len(retrieval_comparison_df)}")

print("-" * 80)

print("Source-aware retrieval quality summary:")
print(source_aware_retrieval_evaluation_df["retrieval_quality"].value_counts(dropna=False))

Saved source-aware evaluation to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_aware_retrieval_evaluation_summary.csv
Source-aware evaluation exists: True
Source-aware evaluation rows: 8
--------------------------------------------------------------------------------
Saved retrieval comparison to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\retrieval_baseline_vs_source_aware_comparison.csv
Retrieval comparison exists: True
Retrieval comparison rows: 8
--------------------------------------------------------------------------------
Source-aware retrieval quality summary:
retrieval_quality
strong    8
Name: count, dtype: int64


## Notebook 05 Final Checkpoint

This notebook expanded the RAG system from a single-source prototype into a stronger multi-source retrieval system.

Completed work:

- Loaded all processed chunks from five authoritative public sources
- Validated the multi-source chunk table
- Generated sentence-transformer embeddings for all chunks
- Saved embedding matrix and metadata locally
- Created a multi-source Chroma vector collection
- Inserted 856 chunks into the vector store
- Built a reusable multi-source retrieval function
- Evaluated retrieval quality across AI Act, GDPR, and NIST questions
- Added source-aware reranking to improve source precision
- Compared baseline retrieval with source-aware retrieval

The source-aware retrieval layer improved all test cases to strong retrieval quality.

The next project step is to build the final LLM-assisted RAG answer generation layer.

In [32]:
# Final Notebook 05 checkpoint
# Fixed version: uses multisource_chunks_df directly for source count

notebook_05_checkpoint = {
    "sources_in_inventory": int(multisource_chunks_df["source_id"].nunique()),
    "multisource_chunk_rows": int(len(multisource_chunks_df)),
    "embedding_model": embedding_model_name,
    "embedding_matrix_shape": multisource_embedding_matrix.shape,
    "chroma_collection_name": multisource_collection_name,
    "chroma_collection_count": int(multisource_collection.count()),
    "baseline_retrieval_tests": int(len(retrieval_evaluation_df)),
    "baseline_strong_count": int((retrieval_evaluation_df["retrieval_quality"] == "strong").sum()),
    "baseline_acceptable_count": int((retrieval_evaluation_df["retrieval_quality"] == "acceptable").sum()),
    "baseline_needs_review_count": int((retrieval_evaluation_df["retrieval_quality"] == "needs_review").sum()),
    "source_aware_retrieval_tests": int(len(source_aware_retrieval_evaluation_df)),
    "source_aware_strong_count": int((source_aware_retrieval_evaluation_df["retrieval_quality"] == "strong").sum()),
    "source_aware_acceptable_count": int((source_aware_retrieval_evaluation_df["retrieval_quality"] == "acceptable").sum()),
    "source_aware_needs_review_count": int((source_aware_retrieval_evaluation_df["retrieval_quality"] == "needs_review").sum()),
    "source_aware_summary_file_exists": source_aware_evaluation_output_path.exists(),
    "retrieval_comparison_file_exists": retrieval_comparison_output_path.exists(),
}

print("Notebook 05 final checkpoint completed.")
print("-" * 80)

for key, value in notebook_05_checkpoint.items():
    print(f"{key}: {value}")

Notebook 05 final checkpoint completed.
--------------------------------------------------------------------------------
sources_in_inventory: 5
multisource_chunk_rows: 856
embedding_model: sentence-transformers/all-MiniLM-L6-v2
embedding_matrix_shape: (856, 384)
chroma_collection_name: responsible_ai_rag_multisource_v1
chroma_collection_count: 856
baseline_retrieval_tests: 8
baseline_strong_count: 5
baseline_acceptable_count: 3
baseline_needs_review_count: 0
source_aware_retrieval_tests: 8
source_aware_strong_count: 8
source_aware_acceptable_count: 0
source_aware_needs_review_count: 0
source_aware_summary_file_exists: True
retrieval_comparison_file_exists: True


## Next Notebook

The next notebook will be:

`06_llm_rag_assistant_and_guardrails.ipynb`

The goal of the next notebook is to build the final recruiter-friendly RAG assistant.

The next notebook will:

1. Load the multi-source Chroma vector store.
2. Use source-aware retrieval.
3. Format retrieved context with source citations.
4. Create a responsible-use system prompt.
5. Generate answers using an LLM.
6. Add guardrails for insufficient context and legal-advice limitations.
7. Save example RAG answers for the project report.
8. Prepare the assistant logic for a future Streamlit interface.

The next stage connects the evaluated retrieval layer to LLM-assisted answer generation.